# Baseline comparison (A vs B vs C)

Ties sub-baselines A (descriptive clustering), B (topological PH on fastText CC-300),
and C (cross-lingual MUSE alignment) into a single comparison notebook with
paper-ready figures and a qualitative sanity check of the four pre-registered predictions.

**No new analyses are run here** — all figures are assembled from result artifacts
already written by notebooks baseline_A, B, and C.

## Setup

In [1]:
# Ensure the repo root is on sys.path (mirrors baseline A/B/C pattern)
import sys
import pathlib

_NOTEBOOK_DIR = pathlib.Path(__file__).resolve().parent if '__file__' in dir() else pathlib.Path('.').resolve()
_REPO_ROOT = _NOTEBOOK_DIR.parent if _NOTEBOOK_DIR.name == 'notebooks' else _NOTEBOOK_DIR

if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

print(f'Repo root: {_REPO_ROOT}')

Repo root: /home/anna/ph-project-baseline-comparison


In [2]:
import warnings

import matplotlib
matplotlib.use('Agg')  # non-interactive backend for headless execution
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# ── Constants ──────────────────────────────────────────────────────────────
REPO_ROOT    = _REPO_ROOT
RESULTS_DIR  = REPO_ROOT / 'results'
FIGURES_DIR  = RESULTS_DIR / 'figures'
A_DIR        = RESULTS_DIR / 'baseline_A'
B_DIR        = RESULTS_DIR / 'baseline_B'
C_DIR        = RESULTS_DIR / 'baseline_C'

LANGUAGES    = ['en', 'es', 'ru']
DOMAINS      = ['color', 'emotion', 'kinship']

LANG_LABELS  = {'en': 'English', 'es': 'Spanish', 'ru': 'Russian'}
DOMAIN_LABELS = {'color': 'Color', 'emotion': 'Emotion', 'kinship': 'Kinship'}

# ── Ensure figures output directory exists ──────────────────────────────────
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.options.display.max_columns = 40
pd.options.display.width = 200

print('Setup complete.')
print(f'Figures will be written to: {FIGURES_DIR}')

Setup complete.
Figures will be written to: /home/anna/ph-project-baseline-comparison/results/figures


## Load feature tables

Load the three sub-baseline result tables and assemble `results/baseline_summary.csv`:
a single flat table with one row per (language, domain) combining
A's cluster statistics, all of B's Kushnareva features, and C's cross-lingual distances.

In [3]:
# ── Baseline A ─────────────────────────────────────────────────────────────
df_a = pd.read_csv(A_DIR / 'summary.csv')
df_a = df_a.rename(columns={
    'best_k': 'a_best_k',
    'best_silhouette': 'a_best_silhouette',
})
# keep only the cols we need from A
df_a = df_a[['language', 'domain', 'a_best_k', 'a_best_silhouette']]

# ── Baseline B ─────────────────────────────────────────────────────────────
df_b = pd.read_csv(B_DIR / 'features.csv')
# prefix all feature cols with 'b_'
b_feature_cols = [c for c in df_b.columns if c not in ('language', 'domain')]
df_b = df_b.rename(columns={c: f'b_{c}' for c in b_feature_cols})

# ── Baseline C ─────────────────────────────────────────────────────────────
df_c_oov   = pd.read_csv(C_DIR / 'oov_rates.csv')
df_c_pred2 = pd.read_csv(C_DIR / 'pred2.csv')

# Normalise lang column in pred2: 'EN-RU' -> both ('en','ru') rows get the distance
# For each (lang, domain), compute the MEAN of the two pairs that involve that lang.
def _pred2_per_lang(pred2: pd.DataFrame) -> pd.DataFrame:
    """Pivot pred2 so each (language, domain) gets mean_c_dist and a c_warn flag."""
    records = []
    for (lang_pair, domain), grp in pred2.groupby(['lang_pair', 'domain']):
        l1, l2 = lang_pair.split('-')
        dist   = grp['mean_distance'].values[0]
        marker = grp['marker'].values[0]
        warn   = (str(marker).strip() == '\u26a0') if pd.notna(marker) else False
        for lang in (l1.lower(), l2.lower()):
            records.append({'language': lang, 'domain': domain,
                            '_dist': dist, '_warn': warn})
    df = pd.DataFrame(records)
    # mean distance across the (up to) 2 pairs per (lang, domain)
    agg = df.groupby(['language', 'domain']).agg(
        c_mean_cross_dist=('_dist', 'mean'),
        c_warn_any=('_warn', 'any'),
    ).reset_index()
    agg['c_warn'] = agg['c_warn_any'].map({True: '\u26a0', False: ''})
    return agg[['language', 'domain', 'c_mean_cross_dist', 'c_warn']]

df_c_lang = _pred2_per_lang(df_c_pred2)

# OOV tier from baseline C
df_c_tier = df_c_oov[['lang', 'domain', 'tier']].rename(columns={'lang': 'language', 'tier': 'c_oov_tier'})

print('Baseline A shape:', df_a.shape)
print('Baseline B shape:', df_b.shape)
print('Baseline C (per-lang distances) shape:', df_c_lang.shape)
print('Baseline C (OOV tier) shape:', df_c_tier.shape)

Baseline A shape: (9, 4)
Baseline B shape: (9, 26)
Baseline C (per-lang distances) shape: (9, 4)
Baseline C (OOV tier) shape: (9, 3)


In [4]:
# ── Assemble baseline_summary.csv ─────────────────────────────────────────
# Join on (language, domain); column order: language, domain, A cols, B cols, C cols
df_summary = (
    df_a
    .merge(df_b,      on=['language', 'domain'], how='left')
    .merge(df_c_lang, on=['language', 'domain'], how='left')
    .merge(df_c_tier, on=['language', 'domain'], how='left')
)

# Sort for a canonical order
df_summary = df_summary.sort_values(['domain', 'language']).reset_index(drop=True)

out_path = RESULTS_DIR / 'baseline_summary.csv'
df_summary.to_csv(out_path, index=False)
print(f'Written: {out_path}  ({len(df_summary)} rows)')

# Display the full table
df_summary

Written: /home/anna/ph-project-baseline-comparison/results/baseline_summary.csv  (9 rows)


,language,domain,a_best_k,a_best_silhouette,b_h0_s,b_h0_m,b_h0_v,b_h0_e,b_h0_n_d_m_t0.25,b_h0_n_d_m_t0.5,b_h0_n_d_m_t0.75,b_h0_n_b_l_t0.25,b_h0_n_b_l_t0.5,b_h0_n_b_l_t0.75,b_h0_t_b,b_h0_t_d,b_h1_s,b_h1_m,b_h1_v,b_h1_e,b_h1_n_d_m_t0.25,b_h1_n_d_m_t0.5,b_h1_n_d_m_t0.75,b_h1_n_b_l_t0.25,b_h1_n_b_l_t0.5,b_h1_n_b_l_t0.75,b_h1_t_b,b_h1_t_d,c_mean_cross_dist,c_warn,c_oov_tier
0,en,color,8,0.0721,2.402799,0.240280,0.070160,2.259548,4.0,0.0,0.0,10.0,10.0,10.0,0.0,0.353995,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.48620,,1
1,es,color,6,0.0520,2.346132,0.234613,0.053828,2.274858,6.0,0.0,0.0,10.0,10.0,10.0,0.0,0.306438,0.008816,0.008816,0.000000,0.000000,1.0,0.0,0.0,0.0,1.0,1.0,0.303743,0.312558,0.48830,,1
2,ru,color,4,0.0979,1.727206,0.157019,0.030540,2.379182,0.0,0.0,0.0,11.0,11.0,11.0,0.0,0.202376,0.034525,0.017263,0.004999,0.650616,1.0,0.0,0.0,2.0,2.0,2.0,0.243137,0.265398,0.48160,,1
3,en,emotion,3,-0.0284,7.078964,0.416410,0.081801,2.813974,17.0,2.0,0.0,17.0,17.0,17.0,0.0,0.596145,0.169686,0.024241,0.022653,1.522815,7.0,7.0,0.0,0.0,0.0,7.0,0.547855,0.616293,0.57710,,1
4,es,emotion,2,0.0074,8.336538,0.396978,0.108744,3.005472,19.0,1.0,0.0,21.0,21.0,21.0,0.0,0.723437,0.289983,0.048330,0.028773,1.608096,6.0,3.0,0.0,0.0,6.0,6.0,0.487190,0.586815,0.58560,,1
5,ru,emotion,2,0.1037,7.267117,0.403729,0.129055,2.838898,17.0,3.0,0.0,18.0,18.0,18.0,0.0,0.689067,0.226793,0.037799,0.025904,1.567193,6.0,4.0,0.0,0.0,5.0,6.0,0.475854,0.563020,0.60230,,1
6,en,kinship,2,-0.0525,4.220058,0.162310,0.028753,3.242051,0.0,0.0,0.0,26.0,26.0,26.0,0.0,0.218654,1.063089,0.055952,0.030548,2.776006,11.0,0.0,0.0,18.0,19.0,19.0,0.144890,0.254304,0.39095,⚠,2
7,es,kinship,2,0.0301,6.713316,0.216559,0.074165,3.382942,6.0,0.0,0.0,31.0,31.0,31.0,0.0,0.481171,1.032937,0.046952,0.037938,2.740691,20.0,0.0,0.0,10.0,22.0,22.0,0.190071,0.323913,0.42145,⚠,1
8,ru,kinship,2,0.1771,7.777062,0.250873,0.058365,3.407583,14.0,0.0,0.0,31.0,31.0,31.0,0.0,0.388121,0.858585,0.047699,0.032139,2.622793,18.0,0.0,0.0,1.0,18.0,18.0,0.274595,0.380902,0.43540,⚠,2


## Cross-method comparison (A side-by-side B per domain)

For each domain, a 3-column × 2-row panel:
- **Top row**: A dendrograms (English, Spanish, Russian)
- **Bottom row**: B barcodes (English, Spanish, Russian)

Figures are assembled from existing PNG exports in `results/baseline_A/` and
`results/baseline_B/` using `matplotlib.image.imread()` — no re-computation.

In [5]:
def save_fig(fig, stem: str) -> None:
    """Export a figure as both .pdf (vector) and .png (300 dpi) to FIGURES_DIR."""
    pdf_path = FIGURES_DIR / f'{stem}.pdf'
    png_path = FIGURES_DIR / f'{stem}.png'
    fig.savefig(pdf_path, bbox_inches='tight')
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    print(f'  Saved {pdf_path.name}  +  {png_path.name}')


for domain in DOMAINS:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(
        f'Fig 1 — {DOMAIN_LABELS[domain]}: Baseline A (dendrogram) vs Baseline B (barcode)',
        fontsize=13, fontweight='bold'
    )

    for col, lang in enumerate(LANGUAGES):
        # Row 0: A dendrograms
        img_a = mpimg.imread(str(A_DIR / 'dendrograms' / f'{lang}_{domain}.png'))
        axes[0, col].imshow(img_a)
        axes[0, col].axis('off')
        axes[0, col].set_title(f'{LANG_LABELS[lang]} — dendrogram (A)', fontsize=10)

        # Row 1: B barcodes (one combined PNG per domain — show same image,
        # crop or use the full image in centre column only for a clean layout)
        if col == 1:  # centre column: show the combined B barcode
            img_b = mpimg.imread(str(B_DIR / 'barcodes' / f'{domain}.png'))
            axes[1, col].imshow(img_b)
            axes[1, col].axis('off')
            axes[1, col].set_title(f'Barcodes EN / ES / RU — {DOMAIN_LABELS[domain]} (B)', fontsize=10)
        else:
            axes[1, col].axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    stem = f'fig1_AB_{domain}'
    save_fig(fig, stem)
    plt.close(fig)
    print(f'  fig1_AB_{domain} — done')

print('\nAll fig1 panels complete.')

  Saved fig1_AB_color.pdf  +  fig1_AB_color.png
  fig1_AB_color — done


  Saved fig1_AB_emotion.pdf  +  fig1_AB_emotion.png
  fig1_AB_emotion — done


  Saved fig1_AB_kinship.pdf  +  fig1_AB_kinship.png
  fig1_AB_kinship — done

All fig1 panels complete.


## Cross-linguistic comparison (B side-by-side C per domain)

For each domain:
- **fig2_BC_\<domain\>**: intra-lingual B barcodes (EN/RU/ES) next to the joint
  cross-lingual C barcode and C dendrogram.
- **fig3_pred2_divergence**: bar chart of mean cross-lingual cosine distances,
  re-rendered from `pred2.csv` (not copied from C).

In [6]:
for domain in DOMAINS:
    # Layout: 2 rows × 3 cols
    # Row 0, col 0-2: B barcodes (combined panel) + C barcode + C dendrogram
    # Row 1, col 0-2: C heatmaps (EN-ES, EN-RU, RU-ES)
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(
        f'Fig 2 — {DOMAIN_LABELS[domain]}: Baseline B (intra-lingual PH) vs Baseline C (cross-lingual)',
        fontsize=13, fontweight='bold'
    )

    # Row 0, col 0: B barcodes (combined)
    img_b = mpimg.imread(str(B_DIR / 'barcodes' / f'{domain}.png'))
    axes[0, 0].imshow(img_b)
    axes[0, 0].axis('off')
    axes[0, 0].set_title('B barcodes (EN / ES / RU)', fontsize=10)

    # Row 0, col 1: C barcode (joint cloud)
    img_cb = mpimg.imread(str(C_DIR / 'barcodes' / f'{domain}.png'))
    axes[0, 1].imshow(img_cb)
    axes[0, 1].axis('off')
    axes[0, 1].set_title('C barcode (joint cloud)', fontsize=10)

    # Row 0, col 2: C dendrogram (joint, colored by language)
    img_cd = mpimg.imread(str(C_DIR / 'dendrograms' / f'{domain}.png'))
    axes[0, 2].imshow(img_cd)
    axes[0, 2].axis('off')
    axes[0, 2].set_title('C dendrogram (joint, colored by language)', fontsize=10)

    # Row 1: C heatmaps
    pairs = ['en-es', 'en-ru', 'ru-es']
    pair_labels = ['EN vs ES', 'EN vs RU', 'RU vs ES']
    for col, (pair, label) in enumerate(zip(pairs, pair_labels)):
        img_h = mpimg.imread(str(C_DIR / 'heatmaps' / f'{domain}_{pair}.png'))
        axes[1, col].imshow(img_h)
        axes[1, col].axis('off')
        axes[1, col].set_title(f'C heatmap: {label}', fontsize=10)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    stem = f'fig2_BC_{domain}'
    save_fig(fig, stem)
    plt.close(fig)
    print(f'  fig2_BC_{domain} — done')

print('\nAll fig2 panels complete.')

  Saved fig2_BC_color.pdf  +  fig2_BC_color.png
  fig2_BC_color — done


  Saved fig2_BC_emotion.pdf  +  fig2_BC_emotion.png
  fig2_BC_emotion — done


  Saved fig2_BC_kinship.pdf  +  fig2_BC_kinship.png
  fig2_BC_kinship — done

All fig2 panels complete.


In [7]:
# ── fig3: re-render pred2 divergence bar chart from pred2.csv ──────────────
df_p2 = pd.read_csv(C_DIR / 'pred2.csv')

# Compute x positions: group by domain, then by lang_pair within domain
domain_order = DOMAINS
pair_order   = ['EN-RU', 'EN-ES', 'RU-ES']
pair_colors  = {'EN-RU': '#4e79a7', 'EN-ES': '#f28e2b', 'RU-ES': '#e15759'}

fig, ax = plt.subplots(figsize=(10, 5))

n_domains = len(domain_order)
n_pairs   = len(pair_order)
group_w   = 0.8
bar_w     = group_w / n_pairs

x_base = np.arange(n_domains)

for i, pair in enumerate(pair_order):
    offsets = x_base + (i - (n_pairs - 1) / 2) * bar_w
    heights = []
    has_warn = []
    for domain in domain_order:
        row = df_p2[(df_p2['lang_pair'] == pair) & (df_p2['domain'] == domain)]
        if len(row) == 0:
            heights.append(0.0)
            has_warn.append(False)
        else:
            heights.append(float(row['mean_distance'].values[0]))
            marker = row['marker'].values[0]
            has_warn.append(pd.notna(marker) and str(marker).strip() == '\u26a0')

    bars = ax.bar(offsets, heights, width=bar_w * 0.9,
                  color=pair_colors[pair], label=pair, alpha=0.85)

    # Add ⚠ annotation above bars with warnings
    for bar, warn in zip(bars, has_warn):
        if warn:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                '\u26a0',
                ha='center', va='bottom', fontsize=9, color='#c00'
            )

ax.set_xticks(x_base)
ax.set_xticklabels([DOMAIN_LABELS[d] for d in domain_order], fontsize=12)
ax.set_ylabel('Mean cross-lingual cosine distance', fontsize=11)
ax.set_title('Fig 3 — Prediction P2: cross-lingual divergence by domain and language pair', fontsize=12)
ax.legend(title='Language pair', fontsize=10)
ax.set_ylim(0, ax.get_ylim()[1] * 1.12)
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
save_fig(fig, 'fig3_pred2_divergence')
plt.close(fig)
print('fig3_pred2_divergence — done')

  Saved fig3_pred2_divergence.pdf  +  fig3_pred2_divergence.png
fig3_pred2_divergence — done


## Pre-registered predictions — qualitative sanity check

Four predictions were registered before any analysis (see `README.md`).  We evaluate
each here against the static-embedding baselines. Formal statistical tests are deferred
to Phase 4 (`ph-project-bt5`).

| ID | Prediction | Baseline verdict |
|----|------------|------------------|
| P1 | Russian color H0 shows extra structure (blue split) vs EN/ES | see §P1 below |
| P2 | Cross-lingual distance: emotion > color | see §P2 below |
| P3 | Russian kinship is topologically more complex (h0_s / h1_s) than EN | see §P3 below |
| P4 | mBERT attention heads specialise differently across languages | out of scope for baseline |

In [8]:
# ── P1: Russian blue split ──────────────────────────────────────────────────
# Key feature: h0_n_d_m_t0.25 = number of H0 bars with death > mean, above threshold 0.25
# This roughly counts how many distinct connected-component merges are "significant".

df_b_full = pd.read_csv(B_DIR / 'features.csv')

color_rows = df_b_full[df_b_full['domain'] == 'color'][['language', 'h0_n_d_m_t0.25', 'h0_s', 'h0_m']].copy()
color_rows = color_rows.set_index('language').loc[LANGUAGES]

print('=== P1: Color-domain H0 features ===')
print(color_rows.to_string())
print()

ru_h0_bars = color_rows.loc['ru', 'h0_n_d_m_t0.25']
en_h0_bars = color_rows.loc['en', 'h0_n_d_m_t0.25']
es_h0_bars = color_rows.loc['es', 'h0_n_d_m_t0.25']

print(f'h0_n_d_m_t0.25: EN={en_h0_bars:.0f}, ES={es_h0_bars:.0f}, RU={ru_h0_bars:.0f}')

if ru_h0_bars > en_h0_bars and ru_h0_bars > es_h0_bars:
    p1_verdict = 'signal present in baseline'
    p1_detail = (f'RU has {ru_h0_bars:.0f} significant H0 bars at threshold 0.25 '
                 f'vs EN={en_h0_bars:.0f} and ES={es_h0_bars:.0f}, '
                 f'consistent with the expected finer color partition.')
elif ru_h0_bars == en_h0_bars and ru_h0_bars == es_h0_bars:
    p1_verdict = 'ambiguous — awaits mBERT'
    p1_detail = (f'All three languages show {ru_h0_bars:.0f} bars at threshold 0.25. '
                 f'Static embeddings do not capture the blue split; mBERT needed.')
else:
    p1_verdict = 'signal absent in baseline'
    p1_detail = (f'RU shows {ru_h0_bars:.0f} significant H0 bars, '
                 f'no more than EN ({en_h0_bars:.0f}) or ES ({es_h0_bars:.0f}). '
                 f'Static embeddings do not distinguish the Russian blue partition.')

print(f'\nP1 verdict: {p1_verdict}')
print(f'P1 detail:  {p1_detail}')

=== P1: Color-domain H0 features ===
          h0_n_d_m_t0.25      h0_s      h0_m
language                                    
en                   4.0  2.402799  0.240280
es                   6.0  2.346132  0.234613
ru                   0.0  1.727206  0.157019

h0_n_d_m_t0.25: EN=4, ES=6, RU=0

P1 verdict: signal absent in baseline
P1 detail:  RU shows 0 significant H0 bars, no more than EN (4) or ES (6). Static embeddings do not distinguish the Russian blue partition.


In [9]:
# ── fig4: P1 focused panel — B color barcodes for EN, RU, ES ─────────────
# Show the full combined B barcode panel for color and annotate bar counts.

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    'Fig 4 — Prediction P1: Russian blue split\n'
    'Baseline B color barcodes (EN / ES / RU) + H0 bar-count comparison',
    fontsize=12, fontweight='bold'
)

# Left: B color barcodes
img_b_color = mpimg.imread(str(B_DIR / 'barcodes' / 'color.png'))
axes[0].imshow(img_b_color)
axes[0].axis('off')
axes[0].set_title('B barcodes — Color (EN / ES / RU)', fontsize=10)

# Right: bar chart of h0_n_d_m_t0.25 for color
color_df = df_b_full[df_b_full['domain'] == 'color'][['language', 'h0_n_d_m_t0.25', 'h0_s']]
color_df = color_df.set_index('language').loc[LANGUAGES]

x = np.arange(len(LANGUAGES))
w = 0.35
b1 = axes[1].bar(x - w/2, color_df['h0_n_d_m_t0.25'], width=w, label='h0_n_d_m_t0.25', color='#4e79a7', alpha=0.85)
b2 = axes[1].bar(x + w/2, color_df['h0_s'],            width=w, label='h0_s (sum of persistence)', color='#f28e2b', alpha=0.85)

axes[1].set_xticks(x)
axes[1].set_xticklabels([LANG_LABELS[l] for l in LANGUAGES], fontsize=11)
axes[1].set_ylabel('Value', fontsize=10)
axes[1].set_title('Color domain: H0 bar count (t=0.25) and persistence sum', fontsize=10)
axes[1].legend(fontsize=9)
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

# Annotate values on bars
for bar in list(b1) + list(b2):
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, h + 0.02,
                 f'{h:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout(rect=[0, 0, 1, 0.90])
save_fig(fig, 'fig4_pred1_russian_blue')
plt.close(fig)
print('fig4_pred1_russian_blue — done')

  Saved fig4_pred1_russian_blue.pdf  +  fig4_pred1_russian_blue.png
fig4_pred1_russian_blue — done


### P1 — Russian blue split

**Question:** Does baseline B's Russian color H0 show more persistent structure
(extra connected component) than English or Spanish?

**Key feature:** `h0_n_d_m_t0.25` — number of H0 bars whose death exceeds the
mean death time by at least 0.25 (proxy for "significant" cluster splits).

**Supporting numbers** (from `baseline_B/features.csv`):
- English: `h0_n_d_m_t0.25 = 4`
- Spanish: `h0_n_d_m_t0.25 = 6`
- Russian: `h0_n_d_m_t0.25 = 0`

**Verdict: signal absent in baseline.** Russian color shows *fewer* significant H0 bars
than English or Spanish at this threshold. The Russian blue/goluboy distinction does not
manifest in fastText CC-300 static embeddings — expected, since static embeddings average
over contexts. The prediction requires mBERT attention-weighted representations to test properly.

In [10]:
# ── P2: Emotion > color divergence ─────────────────────────────────────────
# Compare mean cross-lingual distance for emotion vs color per language pair

df_p2_display = df_p2.copy()
print('=== P2: Cross-lingual distances (from pred2.csv) ===')
print(df_p2_display.to_string(index=False))
print()

for pair in pair_order:
    sub = df_p2[df_p2['lang_pair'] == pair]
    d_color   = sub[sub['domain'] == 'color']['mean_distance'].values
    d_emotion = sub[sub['domain'] == 'emotion']['mean_distance'].values
    if len(d_color) and len(d_emotion):
        print(f'{pair}: color={d_color[0]:.4f}  emotion={d_emotion[0]:.4f}  '
              f'emotion > color? {d_emotion[0] > d_color[0]}')

# Aggregate
mean_color   = df_p2[df_p2['domain'] == 'color']['mean_distance'].mean()
mean_emotion = df_p2[df_p2['domain'] == 'emotion']['mean_distance'].mean()
print(f'\nMean across all pairs: color={mean_color:.4f}  emotion={mean_emotion:.4f}')

if mean_emotion > mean_color:
    p2_verdict = 'signal present in baseline'
    p2_detail  = (f'Emotion mean cross-lingual distance ({mean_emotion:.4f}) > '
                  f'color ({mean_color:.4f}) across all language pairs.')
else:
    p2_verdict = 'signal absent in baseline'
    p2_detail  = (f'Color ({mean_color:.4f}) >= emotion ({mean_emotion:.4f}) on average; '
                  f'prediction not supported by static-embedding baseline.')

print(f'\nP2 verdict: {p2_verdict}')
print(f'P2 detail:  {p2_detail}')

=== P2: Cross-lingual distances (from pred2.csv) ===
lang_pair  domain  mean_distance marker
    EN-RU   color         0.4795    NaN
    EN-ES   color         0.4929    NaN
    RU-ES   color         0.4837    NaN
    EN-RU emotion         0.5938    NaN
    EN-ES emotion         0.5604    NaN
    RU-ES emotion         0.6108    NaN
    EN-RU kinship         0.4049      ⚠
    EN-ES kinship         0.3770      ⚠
    RU-ES kinship         0.4659      ⚠

EN-RU: color=0.4795  emotion=0.5938  emotion > color? True
EN-ES: color=0.4929  emotion=0.5604  emotion > color? True
RU-ES: color=0.4837  emotion=0.6108  emotion > color? True

Mean across all pairs: color=0.4854  emotion=0.5883

P2 verdict: signal present in baseline
P2 detail:  Emotion mean cross-lingual distance (0.5883) > color (0.4854) across all language pairs.


### P2 — Emotion > color cross-lingual divergence

**Question:** Does baseline C show greater mean cross-lingual cosine distance in the
emotion domain than in the color domain?

**Supporting numbers** (from `baseline_C/pred2.csv`):

| Pair | Color | Emotion |
|------|-------|---------|
| EN-RU | 0.4795 | 0.5938 |
| EN-ES | 0.4929 | 0.5604 |
| RU-ES | 0.4837 | 0.6108 |

Mean across all pairs: **Color ≈ 0.485, Emotion ≈ 0.588**.

**Verdict: signal present in baseline.** Emotion cross-lingual distances are
consistently higher than color across all three language pairs — roughly 0.10 units
higher on average. This is an encouraging baseline signal, though the formal test
awaits Phase 4 (`ph-project-bt5`).

In [11]:
# ── P3: Russian kinship complexity ─────────────────────────────────────────
# Compare RU h0_s and h1_s for kinship against EN and ES

kinship_rows = df_b_full[df_b_full['domain'] == 'kinship'][['language', 'h0_s', 'h1_s']].copy()
kinship_rows = kinship_rows.set_index('language').loc[LANGUAGES]

print('=== P3: Kinship-domain H0/H1 persistence sum ===')
print(kinship_rows.to_string())
print()

ru_h0_s = kinship_rows.loc['ru', 'h0_s']
en_h0_s = kinship_rows.loc['en', 'h0_s']
es_h0_s = kinship_rows.loc['es', 'h0_s']

ru_h1_s = kinship_rows.loc['ru', 'h1_s']
en_h1_s = kinship_rows.loc['en', 'h1_s']
es_h1_s = kinship_rows.loc['es', 'h1_s']

if ru_h0_s > en_h0_s or ru_h1_s > en_h1_s:
    p3_verdict = 'signal present in baseline'
    p3_detail  = (f'RU kinship: h0_s={ru_h0_s:.3f}, h1_s={ru_h1_s:.3f}; '
                  f'EN: h0_s={en_h0_s:.3f}, h1_s={en_h1_s:.3f}; '
                  f'ES: h0_s={es_h0_s:.3f}, h1_s={es_h1_s:.3f}. '
                  f'Russian kinship shows higher topological complexity on at least one metric.')
else:
    p3_verdict = 'signal absent in baseline'
    p3_detail  = (f'RU kinship h0_s={ru_h0_s:.3f} / h1_s={ru_h1_s:.3f} '
                  f'not notably higher than EN (h0_s={en_h0_s:.3f} / h1_s={en_h1_s:.3f}).')

print(f'P3 verdict: {p3_verdict}')
print(f'P3 detail:  {p3_detail}')

=== P3: Kinship-domain H0/H1 persistence sum ===
              h0_s      h1_s
language                    
en        4.220058  1.063089
es        6.713316  1.032937
ru        7.777062  0.858585

P3 verdict: signal present in baseline
P3 detail:  RU kinship: h0_s=7.777, h1_s=0.859; EN: h0_s=4.220, h1_s=1.063; ES: h0_s=6.713, h1_s=1.033. Russian kinship shows higher topological complexity on at least one metric.


In [12]:
# ── fig5: P3 kinship complexity panel ──────────────────────────────────────

kin_df = df_b_full[df_b_full['domain'] == 'kinship'][['language', 'h0_s', 'h1_s']]
kin_df = kin_df.set_index('language').loc[LANGUAGES]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(
    'Fig 5 — Prediction P3: Russian kinship complexity\n'
    'Baseline B kinship h0_s and h1_s (sum of persistence) by language',
    fontsize=12, fontweight='bold'
)

x = np.arange(len(LANGUAGES))
w = 0.35

# Left: h0_s
b0 = axes[0].bar(x, kin_df['h0_s'], width=w*1.5, color=['#4e79a7', '#f28e2b', '#e15759'], alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels([LANG_LABELS[l] for l in LANGUAGES], fontsize=11)
axes[0].set_ylabel('h0_s (sum of H0 bar lengths)', fontsize=10)
axes[0].set_title('Kinship: H0 persistence sum', fontsize=11)
axes[0].grid(axis='y', linestyle='--', alpha=0.4)
for bar in b0:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

# Right: h1_s
b1 = axes[1].bar(x, kin_df['h1_s'], width=w*1.5, color=['#4e79a7', '#f28e2b', '#e15759'], alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels([LANG_LABELS[l] for l in LANGUAGES], fontsize=11)
axes[1].set_ylabel('h1_s (sum of H1 bar lengths)', fontsize=10)
axes[1].set_title('Kinship: H1 persistence sum (loops)', fontsize=11)
axes[1].grid(axis='y', linestyle='--', alpha=0.4)
for bar in b1:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 0.90])
save_fig(fig, 'fig5_pred3_kinship_complexity')
plt.close(fig)
print('fig5_pred3_kinship_complexity — done')

  Saved fig5_pred3_kinship_complexity.pdf  +  fig5_pred3_kinship_complexity.png
fig5_pred3_kinship_complexity — done


### P3 — Russian kinship complexity

**Question:** Does baseline B show higher topological complexity (h0_s, h1_s) for
Russian kinship than for English?

**Supporting numbers** (from `baseline_B/features.csv`):

| Language | h0_s | h1_s |
|----------|------|------|
| English  | 4.220 | 1.063 |
| Spanish  | 6.713 | 1.033 |
| Russian  | 7.777 | 0.859 |

**Verdict: signal present in baseline.** Russian kinship has the highest h0_s (7.777 vs
EN 4.220 and ES 6.713), indicating greater spread / more persistent connected-component
structure. However, Russian h1_s (0.859) is *lower* than English (1.063) and Spanish (1.033),
meaning Russian kinship topology has fewer loops — the complexity manifests in zero-dimensional
structure rather than cyclic topology. This is a partial signal; the formal test awaits Phase 4.

### P4 — Head specialization across languages

**Verdict: out of scope for baseline.** P4 requires mBERT attention head analysis (Phase 3,
`ph-project-bt5`). The static-embedding baselines carry no attention information. No
qualitative assessment is possible here.

## OOV / coverage caveats

Baseline C uses MUSE supervised aligned fastText vectors which have no subword fallback,
so OOV rates are higher than for CC-300. The notebook applied a three-tier gate:
- Tier 1 (OOV ≤ 10%): proceed normally
- Tier 2 (10% < OOV ≤ 25%): proceed with a ⚠ warning in downstream results
- Tier 3 (OOV > 25%): skip — recorded in `oov_skipped.csv`

In [13]:
df_oov = pd.read_csv(C_DIR / 'oov_rates.csv')
print('=== Baseline C OOV rates ===')
print(df_oov.to_string(index=False))
print()

tier2 = df_oov[df_oov['tier'] == 2]
tier3 = df_oov[df_oov['tier'] == 3]

if len(tier2):
    print('⚠ Tier-2 (warning) entries:')
    print(tier2[['lang', 'domain', 'oov_rate']].to_string(index=False))
    print('  These (lang, domain) cells carry ⚠ markers in pred2.csv.')
    print('  Downstream comparisons for kinship should be read with caution.')
else:
    print('No tier-2 entries.')

print()
df_skipped = pd.read_csv(C_DIR / 'oov_skipped.csv')
if len(df_skipped) > 0:
    print('⛔ Tier-3 (skipped) entries — insufficient coverage for baseline C:')
    print(df_skipped.to_string(index=False))
else:
    print('No tier-3 skips — all (lang, domain) combinations passed the OOV gate.')

=== Baseline C OOV rates ===
lang  domain  n_terms  n_found  oov_rate  tier
  en   color       11       11    0.0000     1
  en emotion       18       18    0.0000     1
  en kinship       27       21    0.2222     2
  es   color       11       11    0.0000     1
  es emotion       22       22    0.0000     1
  es kinship       32       31    0.0312     1
  ru   color       12       12    0.0000     1
  ru emotion       19       19    0.0000     1
  ru kinship       34       28    0.1765     2

⚠ Tier-2 (warning) entries:
lang  domain  oov_rate
  en kinship    0.2222
  ru kinship    0.1765
  These (lang, domain) cells carry ⚠ markers in pred2.csv.
  Downstream comparisons for kinship should be read with caution.

No tier-3 skips — all (lang, domain) combinations passed the OOV gate.


### Coverage notes

Two (lang, domain) combinations fall into tier 2 (10%–25% OOV in MUSE aligned vectors):
- **English kinship**: OOV = 22.2% (6 of 27 terms missing)
- **Russian kinship**: OOV = 17.6% (6 of 34 terms missing)

Spanish kinship (OOV = 3.1%) and all color/emotion rows are tier 1.  
No domain was skipped entirely (tier 3 is empty).

The kinship ⚠ marker in pred2.csv should be carried forward:
P3 conclusions about cross-lingual kinship distances are tentative pending
improved term coverage in the MUSE lexicon (tracked as a discovered-from issue
under ssa.6).

## Discussion

The three static-embedding baselines establish a useful, if limited, foundation for the
main mBERT hypothesis. **Baseline A** shows weak-to-modest cluster structure overall:
color silhouettes sit in 0.05–0.10 across all three languages, while emotion and kinship
silhouettes are highly variable (English emotion = −0.03, English kinship = −0.05 are
essentially borderline-random; Russian emotion = 0.10 and Russian kinship = 0.18 are the
best-defined of any condition). The picture is consistent with English's relatively
polysemous, sparsely-marked kinship vocabulary versus Russian's morphologically richer one.
**Baseline B** adds topological depth: Russian kinship stands out with the highest H0
persistence sum, suggesting its term cloud is more spread and more internally
differentiated than English or Spanish, which is compatible with the richer Russian
kinship vocabulary. **Baseline C** provides the clearest early signal: emotion
cross-lingual distances are consistently about 0.10 cosine units larger than color
distances across all language pairs (P2), supporting the idea that emotion vocabulary is
more culturally variable than color vocabulary even in static embeddings.

What the baselines *cannot* tell us: whether the topological differences reflect
genuine attentional/semantic distinctions or artifacts of vocabulary size and OOV rates.
The Russian blue split (P1) and head specialization (P4) require contextualized
representations — P1 because *синий/голубой* share a distributional neighborhood in
static fastText, and P4 by definition needs attention heads. **Phase 3 (mBERT,
`ph-project-bt5`)** will extract per-sentence attention matrices and apply the full
Kushnareva feature pipeline to contextualized representations. **Phase 4 (`ph-project-bt5`)**
will apply permutation tests and effect-size estimates to all four predictions simultaneously,
with the present baselines serving as the primary foil for the mBERT results.

In [14]:
# ── Final sanity check: verify all expected outputs exist ──────────────────
import os

expected_files = [
    RESULTS_DIR / 'baseline_summary.csv',
]
for domain in DOMAINS:
    for stem in [f'fig1_AB_{domain}', f'fig2_BC_{domain}']:
        expected_files.append(FIGURES_DIR / f'{stem}.pdf')
        expected_files.append(FIGURES_DIR / f'{stem}.png')
for stem in ['fig3_pred2_divergence', 'fig4_pred1_russian_blue', 'fig5_pred3_kinship_complexity']:
    expected_files.append(FIGURES_DIR / f'{stem}.pdf')
    expected_files.append(FIGURES_DIR / f'{stem}.png')

missing = []
for f in expected_files:
    if not f.exists():
        missing.append(str(f))

if missing:
    print('MISSING FILES:')
    for m in missing:
        print(f'  {m}')
    raise FileNotFoundError(f'{len(missing)} expected output files are missing.')
else:
    print(f'All {len(expected_files)} expected output files are present.')
    print()
    print('baseline_summary.csv rows:', len(pd.read_csv(RESULTS_DIR / 'baseline_summary.csv')))
    print()
    print('Figures written:')
    for f in sorted(FIGURES_DIR.iterdir()):
        print(f'  {f.name}  ({f.stat().st_size // 1024} kB)')

All 19 expected output files are present.

baseline_summary.csv rows: 9

Figures written:
  fig1_AB_color.pdf  (121 kB)
  fig1_AB_color.png  (588 kB)
  fig1_AB_emotion.pdf  (107 kB)
  fig1_AB_emotion.png  (540 kB)
  fig1_AB_kinship.pdf  (96 kB)
  fig1_AB_kinship.png  (480 kB)
  fig2_BC_color.pdf  (205 kB)
  fig2_BC_color.png  (1203 kB)
  fig2_BC_emotion.pdf  (193 kB)
  fig2_BC_emotion.png  (1202 kB)
  fig2_BC_kinship.pdf  (194 kB)
  fig2_BC_kinship.png  (904 kB)
  fig3_pred2_divergence.pdf  (16 kB)
  fig3_pred2_divergence.png  (124 kB)
  fig4_pred1_russian_blue.pdf  (56 kB)
  fig4_pred1_russian_blue.png  (282 kB)
  fig5_pred3_kinship_complexity.pdf  (27 kB)
  fig5_pred3_kinship_complexity.png  (163 kB)
